# grabs the customers from customer-orders.csv and shows the total

In [ ]:
from pyspark.sql import SparkSession, functions as func
from pyspark.sql.types import StructType, StructField, FloatType, IntegerType


spark = SparkSession.builder.appName("test").getOrCreate()


schema = StructType([
    StructField('customerId', IntegerType()),
    StructField('orderId', IntegerType()),
    StructField('total', FloatType()),
])

customer = spark.read.csv('../../customer-orders.csv', schema=schema, header=False)

# have to specify agg so we can make an alias for the total amount a customer spent
total = customer.groupBy('customerId').agg(func.sum('total').alias("total_spent"))



total.orderBy('total_spent', ascending = False).show()

spark.stop()



+----------+------------------+
|customerId|       total_spent|
+----------+------------------+
|        68| 6375.450028181076|
|        73| 6206.199985742569|
|        39| 6193.109993815422|
|        54| 6065.390002984554|
|        71| 5995.659991919994|
|         2| 5994.589979887009|
|        97| 5977.190007060766|
|        46| 5963.110011339188|
|        42| 5696.840004444122|
|        59| 5642.890004396439|
|        41| 5637.619991332293|
|         0| 5524.950008839369|
|         8|5517.2399980425835|
|        85|  5503.42998456955|
|        61| 5497.479998707771|
|        32| 5496.049998283386|
|        58| 5437.730004191399|
|        63| 5415.150004655123|
|        15| 5413.510010659695|
|         6| 5397.880012750626|
+----------+------------------+
only showing top 20 rows


In [12]:
# using spark sql instead of using dataframe to do the groupby and agg

from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, FloatType, IntegerType


spark = SparkSession.builder.appName("sql").getOrCreate()

schema = StructType([
    StructField('customerId', IntegerType()),
    StructField('orderId', IntegerType()),
    StructField('total', FloatType()),
])


customer = spark.read.csv('../../customer-orders.csv', schema=schema, header=False)

customer.createOrReplaceTempView('orders')

customers = spark.sql("""SELECT customerId, SUM(total) AS total_spent FROM orders GROUP BY customerId ORDER BY total_spent DESC
""")

customers.show()
spark.stop()

+----------+------------------+
|customerId|       total_spent|
+----------+------------------+
|        68| 6375.450028181076|
|        73| 6206.199985742569|
|        39| 6193.109993815422|
|        54| 6065.390002984554|
|        71| 5995.659991919994|
|         2| 5994.589979887009|
|        97| 5977.190007060766|
|        46| 5963.110011339188|
|        42| 5696.840004444122|
|        59| 5642.890004396439|
|        41| 5637.619991332293|
|         0| 5524.950008839369|
|         8|5517.2399980425835|
|        85|  5503.42998456955|
|        61| 5497.479998707771|
|        32| 5496.049998283386|
|        58| 5437.730004191399|
|        63| 5415.150004655123|
|        15| 5413.510010659695|
|         6| 5397.880012750626|
+----------+------------------+
only showing top 20 rows


# get log info challenge

In [ ]:
from pyspark.sql import SparkSession, functions as func
from pyspark.sql.types import StructType, StructField, FloatType, IntegerType, StringType


spark = SparkSession.builder.appName("app").getOrCreate()
df = spark.read.text("../../access_log.txt")

df2 = df.withColumn("columns", func.split("value", " ")).select(
    func.col("columns")[0].alias("ip"),
    func.col("columns")[6].alias("enpoint"),
    func.col("columns")[8].alias("http_status")
)

df2.show()

df2.createOrReplaceTempView("log_table")

# counting all the totla requests and sorting them by most to least
(df2.groupBy('ip').
 agg(func.count('http_status').
     alias('request_count'))
     .orderBy('request_count', ascending = False).show())

# listing endpoint and counting number of times they are called
(df2.
 groupBy('enpoint').
 agg(func.count('http_status').alias('endpoint_count')).
 orderBy('endpoint_count', ascending = False).show())


# use SQL to list HTTP status codes and count all the requests
spark.sql("SELECT http_status, COUNT(http_status) AS status_count FROM log_table GROUP BY http_status ORDER BY status_count DESC").show()

df.createOrReplaceTempView('logs_table')



+--------------+--------------------+-----------+
|            ip|             enpoint|http_status|
+--------------+--------------------+-----------+
| 66.249.75.159|         /robots.txt|        200|
| 66.249.75.168|              /blog/|        200|
|185.71.216.232|       /wp-login.php|        200|
|54.165.199.171|  /sitemap_index.xml|        200|
|54.165.199.171|   /post-sitemap.xml|        200|
|54.165.199.171|   /page-sitemap.xml|        200|
|54.165.199.171|/category-sitemap...|        200|
|54.165.199.171|              /blog/|        200|
|54.165.199.171|    /orlando-sports/|        200|
|54.165.199.171|             /about/|        200|
|54.165.199.171|          /business/|        200|
|54.165.199.171|        /technology/|        200|
|54.165.199.171| /orlando-headlines/|        200|
|54.165.199.171|     /entertainment/|        200|
|54.165.199.171|            /travel/|        200|
|54.165.199.171|          /national/|        200|
|54.165.199.171|  /dallas-headlines/|        200|


# finding the top 10 most popular movies in u.data

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, FloatType, IntegerType, StringType


spark = SparkSession.builder.appName("log").getOrCreate()

schema = StructType([
    StructField('userId', IntegerType()),
    StructField('itemID', IntegerType()),
    StructField('rating', IntegerType()),
    StructField('timestamp', IntegerType())
])

df = spark.read.option("sep", "\t").schema(schema).csv("../../ml-100k/u.data")

df.sort('rating', ascending=False).show(10)



Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/07/03 14:20:55 WARN Utils: Your hostname, DaniyalsLenovo, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/07/03 14:20:55 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/03 14:20:57 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


+------+------+------+---------+
|userId|itemID|rating|timestamp|
+------+------+------+---------+
|   287|   327|     5|875333916|
|    60|   427|     5|883326620|
|   246|   201|     5|884921594|
|   253|   465|     5|891628467|
|   242|  1137|     5|879741196|
|   122|   387|     5|879270459|
|   249|   241|     5|879641194|
|   160|   234|     5|876861185|
|    99|     4|     5|886519097|
|    25|   181|     5|885853415|
+------+------+------+---------+
only showing top 10 rows
